#Homework #5: model deployment

### Xuanlin Liu (xl3572)

Instructions: Using the F1 dataset, build a predictive model and log it in MLflow and write the ML model predictions into a database.

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

import mlflow
import mlflow.spark
import matplotlib.pyplot as plt
import tempfile
import os
from pyspark.sql.types import StructType, StructField, DoubleType, StringType

**1. 
[20 pts] Create two (2) new tables in your own fatabse where you'll store the predictions from each model for this exercise.**

In [0]:
prediction_schema = StructType([
    StructField("raceId", DoubleType(), True),
    StructField("driverId", DoubleType(), True),
    StructField("actual_position", DoubleType(), True),
    StructField("predicted_position", DoubleType(), True),
    StructField("model_name", StringType(), True)
])

In [0]:
# for linear regression

empty_predictions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/gr5069/xl3572/takehome/linear_regression_predictions")

In [0]:
spark.read.format("delta") \
    .load("/Volumes/gr5069/xl3572/takehome/linear_regression_predictions") \
    .write \
    .mode("overwrite") \
    .saveAsTable("gr5069.xl3572.linear_regression_predictions")

In [0]:
# for decision tree
empty_predictions_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/gr5069/xl3572/takehome/decision_tree_predictions")

In [0]:
spark.read.format("delta") \
    .load("/Volumes/gr5069/xl3572/takehome/decision_tree_predictions") \
    .write \
    .mode("overwrite") \
    .saveAsTable("gr5069.xl3572.decision_tree_predictions")

I created two Delta tables in my schema, `gr5069.xl3572`, to store predictions from each model. The tables are:

Both tables are stored under my `takehome` Volume path and will be used to save the prediction outputs from the two models.

**2. [30 pts] Build two (2) predictive models using MLflow, logging hyperparameters, the model itself, four metrics, and two artifcats. Submit submit your MLflow experiments as part of your assignments**

In [0]:

racesDF = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)

In [0]:
display(racesDF)

In [0]:
resultsDF = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)


In [0]:
display(resultsDF)

In [0]:
# create table
f1DF = resultsDF.join(racesDF.select("raceId", "year", "round", "circuitId"), on="raceId", how="left")

display(f1DF)

In [0]:
modelDF = f1DF.select("raceId", "driverId", "constructorId", "grid", "circuitId", "year", "round",
                      col("positionOrder").alias("label")).dropna()

display(modelDF)

In [0]:
numeric_cols = [
    "raceId", "driverId", "constructorId", "grid",
    "circuitId", "year", "round", "label"
]

for c in numeric_cols:
    modelDF = modelDF.withColumn(c, col(c).cast(DoubleType()))

modelDF = modelDF.dropna()

In [0]:
feature_cols = ["driverId", "constructorId", "grid", "circuitId", "year", "round"]

vecAssembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

finalDF = vecAssembler.transform(modelDF)

In [0]:
trainDF, testDF = finalDF.randomSplit([0.8, 0.2], seed=42)

print(trainDF.count())
print(testDF.count())

In [0]:
# ML logging function
def log_spark_model_run(model_name, model, params, trainDF, testDF):
    with mlflow.start_run(run_name=model_name) as run:
        
        # Fit model
        fitted_model = model.fit(trainDF)
        
        # Predict
        predDF = fitted_model.transform(testDF)
        
        # Log hyperparameters
        for param, value in params.items():
            mlflow.log_param(param, value)
        
        # Log Spark ML model
        mlflow.spark.log_model(
            fitted_model,
            model_name,
            dfs_tmpdir="/Volumes/gr5069/xl3572/takehome/mlflow_tmp"
        )
        
        # Four regression metrics
        evaluator_rmse = RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="rmse"
        )
        
        evaluator_mse = RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="mse"
        )
        
        evaluator_mae = RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="mae"
        )
        
        evaluator_r2 = RegressionEvaluator(
            labelCol="label",
            predictionCol="prediction",
            metricName="r2"
        )
        
        rmse = evaluator_rmse.evaluate(predDF)
        mse = evaluator_mse.evaluate(predDF)
        mae = evaluator_mae.evaluate(predDF)
        r2 = evaluator_r2.evaluate(predDF)
        
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("r2", r2)
        
        # Create artifacts
        temp_dir = tempfile.mkdtemp()
        
        artifactPDF = predDF.select(
            "raceId",
            "driverId",
            col("label").alias("actual_position"),
            col("prediction").alias("predicted_position")
        ).toPandas()
        
        # Artifact 1: predictions CSV
        pred_path = os.path.join(temp_dir, f"{model_name}_predictions.csv")
        artifactPDF.to_csv(pred_path, index=False)
        mlflow.log_artifact(pred_path, "artifacts")
        
        # Artifact 2: actual vs predicted plot
        plt.figure(figsize=(8, 5))
        plt.scatter(
            artifactPDF["actual_position"],
            artifactPDF["predicted_position"],
            alpha=0.5
        )
        plt.xlabel("Actual positionOrder")
        plt.ylabel("Predicted positionOrder")
        plt.title(f"{model_name}: Actual vs Predicted")
        
        plot_path = os.path.join(temp_dir, f"{model_name}_actual_vs_predicted.png")
        plt.savefig(plot_path, bbox_inches="tight")
        plt.close()
        mlflow.log_artifact(plot_path, "artifacts")
        
        print("Model:", model_name)
        print("Run ID:", run.info.run_id)
        print("RMSE:", rmse)
        print("MSE:", mse)
        print("MAE:", mae)
        print("R2:", r2)
        
        return fitted_model, predDF

In [0]:
# linear regression model

lr_params = {
    "maxIter": 10,
    "regParam": 0.1,
    "elasticNetParam": 0.0
}

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=lr_params["maxIter"],
    regParam=lr_params["regParam"],
    elasticNetParam=lr_params["elasticNetParam"]
)

lr_model, lr_predDF = log_spark_model_run(
    model_name="spark_linear_regression_model",
    model=lr,
    params=lr_params,
    trainDF=trainDF,
    testDF=testDF
)

In [0]:
# decision tree model
dt_params = {
    "maxDepth": 5,
    "maxBins": 32
}

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=dt_params["maxDepth"],
    maxBins=dt_params["maxBins"]
)

dt_model, dt_predDF = log_spark_model_run(
    model_name="spark_decision_tree_model",
    model=dt,
    params=dt_params,
    trainDF=trainDF,
    testDF=testDF
)

In [0]:
display(lr_predDF.limit(10))

In [0]:
display(dt_predDF.limit(10))

I built two predictive models using the F1 dataset: a Linear Regression model and a Decision Tree Regressor. Both models predict `positionOrder`.

For each model, I logged the model hyperparameters, the Spark ML model, four regression metrics, and two artifacts in MLflow. 

The four metrics are RMSE, MSE, MAE, and R^2. The two artifacts are a predictions CSV and an actual-vs-predicted plot.


**3. [30 pts] For each model, store its predictions in the corresponding table you created in your own database. Ensure you are using your own database to store your predictions.**

In [0]:
# store predecion for linear regression

linear_predictions_final = lr_predDF.select(
    col("raceId"),
    col("driverId"),
    col("label").alias("actual_position"),
    col("prediction").alias("predicted_position"),
    lit("spark_linear_regression_model").alias("model_name")
)

In [0]:
linear_predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gr5069.xl3572.linear_regression_predictions")

In [0]:
# store predecion for decision tree

decision_tree_predictions_final = dt_predDF.select(
    col("raceId"),
    col("driverId"),
    col("label").alias("actual_position"),
    col("prediction").alias("predicted_position"),
    lit("spark_decision_tree_model").alias("model_name")
)

In [0]:
decision_tree_predictions_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gr5069.xl3572.decision_tree_predictions")

In [0]:
display(spark.table("gr5069.xl3572.linear_regression_predictions").limit(10))

In [0]:
display(spark.table("gr5069.xl3572.decision_tree_predictions").limit(10))

For each model, I stored the prediction output in the corresponding Delta table in my assigned schema, `gr5069.xl3572`.

The Linear Regression model predictions were stored in:
`gr5069.xl3572.linear_regression_predictions`

The Decision Tree model predictions were stored in:
`gr5069.xl3572.decision_tree_predictions`

Each table contains the race ID, driver ID, actual finishing position, predicted finishing position, and model name.

**4. [20 pts] Push your code to GitHub upon completion**